### 加载数据

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from skfolio.datasets import load_sp500_dataset
from skfolio.preprocessing import prices_to_returns
from sklearn.model_selection import train_test_split
#from ml4t.data.providers.qmt_provider import QmtProvider
from sklearn import set_config
set_config(transform_output="pandas")

In [2]:
# 加载数据
prices = pl.read_parquet("hs_funds_prices.parquet").to_pandas()
prices = prices.set_index('timestamp')
prices = prices.ffill()
prices = prices[prices.index.year > 2015]
all_symbols = prices.columns
# 转换为线性收益率
X = prices_to_returns(prices, drop_inceptions_nan=False)
# Inf值检查
inf_cols = X.columns[np.isinf(X).any(axis=0)]
print(inf_cols.tolist())
# 异常收益检查
X = X.drop(columns=(bad := X.columns[(X.abs() > 0.25).any()])); 
print(f"数据异常：收益率超过±30%的列已剔除 {len(bad)} 列 -> {bad.tolist()}")

[]
数据异常：收益率超过±30%的列已剔除 3 列 -> ['161811.SZ', '510030.SH', '511580.SH']


### 预筛选
对初始资产宇宙进行预选择处理，其核心目的是在构建投资组合之前，通过特定的规则筛选出符合条件的资产，从而优化后续的计算效率和模型表现。

In [3]:
import optuna
from skfolio import Population,MultiPeriodPortfolio
from skfolio import RiskMeasure,PerfMeasure,RatioMeasure,ExtraRiskMeasure
from sklearn.pipeline import Pipeline
from skfolio.metrics import make_scorer
from skfolio.optimization import EqualWeighted
from Pre_selection import DropTailCorrelated
from skfolio.pre_selection import SelectKExtremes
from skfolio.pre_selection import DropZeroVariance, DropCorrelated
from skfolio.pre_selection import SelectComplete, SelectNonExpiring, SelectNonDominated
from skfolio.model_selection import WalkForward
from skfolio.model_selection import cross_val_predict
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

### WalkForward交叉验证
- 数据泄露防护与执行延迟控制
通过 purged_size 参数控制训练集和测试集之间的“清洗”间隔，以模拟真实的交易执行延迟。
    - purged_size=0：训练结束与测试开始无缝衔接。
    - purged_size >= 1：在训练集末尾和测试集开头之间丢弃指定数量的观测值。
    - 建议：对于每日定价资产、流动性较差的市场或收盘后结算的数据，建议使用 purged_size >= 1 以更真实地反映执行延迟。

- 训练集扩展与尾部数据处理
    - expand_train=True：后续的训练集将包含所有过去的观测值，而不仅仅是固定长度的窗口。
    - reduce_test=True：即使最后一个测试集的样本数少于 test_size，也会返回该分割。默认情况下，不完整的测试集会被忽略。
#### WalkForward+GridSearch/RandomSearch/Optuna
以最大化样本外的平均指标（如 Mean-CVaR 比率）寻找最优参数。

In [30]:
cv = WalkForward(test_size=252, train_size=int(252*3), purged_size=1, reduce_test=True, expand_train=False)
model = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("nondomin", SelectNonDominated()),
        ("correlate", DropCorrelated()),
        ("optimization", EqualWeighted())
    ])

param_grid = {
    "nondomin__min_n_assets" : [5, 10, 15],
    "nondomin__threshold" : [-0.3, -0.4, -0.5],
    "nondomin__fitness_measures" : [
        [PerfMeasure.MEAN, RiskMeasure.VARIANCE],
        [RatioMeasure.CALMAR_RATIO, RiskMeasure.CVAR],
        [RatioMeasure.CALMAR_RATIO, RiskMeasure.VARIANCE],
        [RatioMeasure.CALMAR_RATIO, RiskMeasure.SEMI_VARIANCE],
        [RatioMeasure.SHARPE_RATIO, RiskMeasure.CVAR],
        [RatioMeasure.SHARPE_RATIO, RiskMeasure.MAX_DRAWDOWN],
        [RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
        [RatioMeasure.SORTINO_RATIO, RiskMeasure.CVAR],
        [RatioMeasure.SORTINO_RATIO, RiskMeasure.VARIANCE],
        [RatioMeasure.SORTINO_RATIO, RiskMeasure.SEMI_VARIANCE]
    ],
    "correlate__threshold" : [0.1, 0.2, 0.3, 0.4, 0.5],
}

In [ ]:
random_search = RandomizedSearchCV(
    estimator=model,
    cv=cv,
    param_distributions=param_grid,
    scoring=make_scorer(PerfMeasure.ANNUALIZED_MEAN),
    n_jobs=12,
    verbose=3,
    n_iter=200,
    refit=False,
)
random_search.fit(X)
print(random_search.best_params_)
print(random_search.best_score_)

#### WalkForward+GridSearch

In [4]:
cv = WalkForward(test_size=252, train_size=int(252*3), purged_size=1, reduce_test=True, expand_train=False)
model = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("nondomin", SelectNonDominated()),
        ("correlate", DropCorrelated()),
        ("optimization", EqualWeighted())
    ])

param_grid = {
    "nondomin__min_n_assets" : [5, 10, 15],
    "nondomin__threshold" : [-0.3, -0.4, -0.5],
    "nondomin__fitness_measures" : [
        [PerfMeasure.MEAN, RiskMeasure.VARIANCE],
        [RatioMeasure.CALMAR_RATIO, RiskMeasure.CVAR],
        [RatioMeasure.CALMAR_RATIO, RiskMeasure.VARIANCE],
        [RatioMeasure.CALMAR_RATIO, RiskMeasure.SEMI_VARIANCE],
        [RatioMeasure.SHARPE_RATIO, RiskMeasure.CVAR],
        [RatioMeasure.SHARPE_RATIO, RiskMeasure.MAX_DRAWDOWN],
        [RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
        [RatioMeasure.SORTINO_RATIO, RiskMeasure.CVAR],
        [RatioMeasure.SORTINO_RATIO, RiskMeasure.VARIANCE],
        [RatioMeasure.SORTINO_RATIO, RiskMeasure.SEMI_VARIANCE]
    ],
    "correlate__threshold" : [0.1, 0.2, 0.3, 0.4, 0.5],
}

In [ ]:
grid_search = GridSearchCV(
    estimator=model,
    cv=cv,
    param_grid=param_grid,
    scoring=make_scorer(PerfMeasure.ANNUALIZED_MEAN),
    n_jobs=8,
    verbose=3,
    refit=False,

)
grid_search.fit(X)
print(grid_search.best_params_)
print(grid_search.best_score_)

#### WalkForward+Optuna

In [4]:
class StopWhenNoImprovement:
    def __init__(self, patience=20, min_delta=1e-4):
        self.patience = patience        # 连续多少次无提升就停止
        self.min_delta = min_delta      # 提升多少才算"有效改善"
        self.best_value = None
        self.no_improve_count = 0

    def __call__(self, study, trial):
        current_value = study.best_value
        
        if self.best_value is None:
            self.best_value = current_value
            return
        
        # 判断是否有有效改善
        if current_value - self.best_value > self.min_delta:
            self.best_value = current_value
            self.no_improve_count = 0
        else:
            self.no_improve_count += 1
        
        if self.no_improve_count >= self.patience:
            print(f"\n连续 {self.patience} 次 trial 无有效改善，提前停止。")
            study.stop()

In [68]:
# MEAN + RiskMeasure
FITNESS_MEASURES = {
    "mean-variance":              [PerfMeasure.MEAN, RiskMeasure.VARIANCE],
    "mean-semivariance":          [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE],
    "mean-std":                   [PerfMeasure.MEAN, RiskMeasure.STANDARD_DEVIATION],
    "mean-semidev":               [PerfMeasure.MEAN, RiskMeasure.SEMI_DEVIATION],
    "mean-mad":                   [PerfMeasure.MEAN, RiskMeasure.MEAN_ABSOLUTE_DEVIATION],
    "mean-cvar":                  [PerfMeasure.MEAN, RiskMeasure.CVAR],
    "mean-evar":                  [PerfMeasure.MEAN, RiskMeasure.EVAR],
    "mean-worst-realization":     [PerfMeasure.MEAN, RiskMeasure.WORST_REALIZATION],
    "mean-cdar":                  [PerfMeasure.MEAN, RiskMeasure.CDAR],
    "mean-maxdd":                 [PerfMeasure.MEAN, RiskMeasure.MAX_DRAWDOWN],
    "mean-avgdd":                 [PerfMeasure.MEAN, RiskMeasure.AVERAGE_DRAWDOWN],
    "mean-edar":                  [PerfMeasure.MEAN, RiskMeasure.EDAR],
    "mean-first-lpm":             [PerfMeasure.MEAN, RiskMeasure.FIRST_LOWER_PARTIAL_MOMENT],
    "mean-ulcer":                 [PerfMeasure.MEAN, RiskMeasure.ULCER_INDEX],
    "mean-gini":                  [PerfMeasure.MEAN, RiskMeasure.GINI_MEAN_DIFFERENCE],
}

In [58]:
# MEAN + RatioMeasure
FITNESS_MEASURES = {
    "mean-sharpe":              [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO],
    "mean-sortino":             [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO],
    "mean-mad":                 [PerfMeasure.MEAN, RatioMeasure.MEAN_ABSOLUTE_DEVIATION_RATIO],
    "mean-first-lpm":           [PerfMeasure.MEAN, RatioMeasure.FIRST_LOWER_PARTIAL_MOMENT_RATIO],
    "mean-var":                 [PerfMeasure.MEAN, RatioMeasure.VALUE_AT_RISK_RATIO],
    "mean-cvar":                [PerfMeasure.MEAN, RatioMeasure.CVAR_RATIO],
    "mean-entropic":            [PerfMeasure.MEAN, RatioMeasure.ENTROPIC_RISK_MEASURE_RATIO],
    "mean-evar":                [PerfMeasure.MEAN, RatioMeasure.EVAR_RATIO],
    "mean-worst-realization":   [PerfMeasure.MEAN, RatioMeasure.WORST_REALIZATION_RATIO],
    "mean-dar":                 [PerfMeasure.MEAN, RatioMeasure.DRAWDOWN_AT_RISK_RATIO],
    "mean-cdar":                [PerfMeasure.MEAN, RatioMeasure.CDAR_RATIO],
    "mean-calmar":              [PerfMeasure.MEAN, RatioMeasure.CALMAR_RATIO],
    "mean-avgdd":               [PerfMeasure.MEAN, RatioMeasure.AVERAGE_DRAWDOWN_RATIO],
    "mean-edar":                [PerfMeasure.MEAN, RatioMeasure.EDAR_RATIO],
    "mean-ulcer":               [PerfMeasure.MEAN, RatioMeasure.ULCER_INDEX_RATIO],
    "mean-gini":                [PerfMeasure.MEAN, RatioMeasure.GINI_MEAN_DIFFERENCE_RATIO],
}

In [ ]:
# SHARPE + RiskMeasure
FITNESS_MEASURES = {
    "sharpe-variance":              [RatioMeasure.SHARPE_RATIO, RiskMeasure.VARIANCE],
    "sharpe-semivariance":          [RatioMeasure.SHARPE_RATIO, RiskMeasure.SEMI_VARIANCE],
    "sharpe-std":                   [RatioMeasure.SHARPE_RATIO, RiskMeasure.STANDARD_DEVIATION],
    "sharpe-semidev":               [RatioMeasure.SHARPE_RATIO, RiskMeasure.SEMI_DEVIATION],
    "sharpe-mad":                   [RatioMeasure.SHARPE_RATIO, RiskMeasure.MEAN_ABSOLUTE_DEVIATION],
    "sharpe-cvar":                  [RatioMeasure.SHARPE_RATIO, RiskMeasure.CVAR],
    "sharpe-evar":                  [RatioMeasure.SHARPE_RATIO, RiskMeasure.EVAR],
    "sharpe-worst-realization":     [RatioMeasure.SHARPE_RATIO, RiskMeasure.WORST_REALIZATION],
    "sharpe-cdar":                  [RatioMeasure.SHARPE_RATIO, RiskMeasure.CDAR],
    "sharpe-maxdd":                 [RatioMeasure.SHARPE_RATIO, RiskMeasure.MAX_DRAWDOWN],
    "sharpe-avgdd":                 [RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
    "sharpe-edar":                  [RatioMeasure.SHARPE_RATIO, RiskMeasure.EDAR],
    "sharpe-first-lpm":             [RatioMeasure.SHARPE_RATIO, RiskMeasure.FIRST_LOWER_PARTIAL_MOMENT],
    "sharpe-ulcer":                 [RatioMeasure.SHARPE_RATIO, RiskMeasure.ULCER_INDEX],
    "sharpe-gini":                  [RatioMeasure.SHARPE_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
}

In [38]:
# SORTINO + RiskMeasure
FITNESS_MEASURES = {
    "sortino-variance":              [RatioMeasure.SORTINO_RATIO, RiskMeasure.VARIANCE],
    "sortino-semivariance":          [RatioMeasure.SORTINO_RATIO, RiskMeasure.SEMI_VARIANCE],
    "sortino-std":                   [RatioMeasure.SORTINO_RATIO, RiskMeasure.STANDARD_DEVIATION],
    "sortino-semidev":               [RatioMeasure.SORTINO_RATIO, RiskMeasure.SEMI_DEVIATION],
    "sortino-mad":                   [RatioMeasure.SORTINO_RATIO, RiskMeasure.MEAN_ABSOLUTE_DEVIATION],
    "sortino-cvar":                  [RatioMeasure.SORTINO_RATIO, RiskMeasure.CVAR],
    "sortino-evar":                  [RatioMeasure.SORTINO_RATIO, RiskMeasure.EVAR],
    "sortino-worst-realization":     [RatioMeasure.SORTINO_RATIO, RiskMeasure.WORST_REALIZATION],
    "sortino-cdar":                  [RatioMeasure.SORTINO_RATIO, RiskMeasure.CDAR],
    "sortino-maxdd":                 [RatioMeasure.SORTINO_RATIO, RiskMeasure.MAX_DRAWDOWN],
    "sortino-avgdd":                 [RatioMeasure.SORTINO_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
    "sortino-edar":                  [RatioMeasure.SORTINO_RATIO, RiskMeasure.EDAR],
    "sortino-first-lpm":             [RatioMeasure.SORTINO_RATIO, RiskMeasure.FIRST_LOWER_PARTIAL_MOMENT],
    "sortino-ulcer":                 [RatioMeasure.SORTINO_RATIO, RiskMeasure.ULCER_INDEX],
    "sortino-gini":                  [RatioMeasure.SORTINO_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
}

In [28]:
# CALMAR + RiskMeasure
FITNESS_MEASURES = {
    "calmar-variance":              [RatioMeasure.CALMAR_RATIO, RiskMeasure.VARIANCE],
    "calmar-semivariance":          [RatioMeasure.CALMAR_RATIO, RiskMeasure.SEMI_VARIANCE],
    "calmar-std":                   [RatioMeasure.CALMAR_RATIO, RiskMeasure.STANDARD_DEVIATION],
    "calmar-semidev":               [RatioMeasure.CALMAR_RATIO, RiskMeasure.SEMI_DEVIATION],
    "calmar-mad":                   [RatioMeasure.CALMAR_RATIO, RiskMeasure.MEAN_ABSOLUTE_DEVIATION],
    "calmar-cvar":                  [RatioMeasure.CALMAR_RATIO, RiskMeasure.CVAR],
    "calmar-evar":                  [RatioMeasure.CALMAR_RATIO, RiskMeasure.EVAR],
    "calmar-worst-realization":     [RatioMeasure.CALMAR_RATIO, RiskMeasure.WORST_REALIZATION],
    "calmar-cdar":                  [RatioMeasure.CALMAR_RATIO, RiskMeasure.CDAR],
    "calmar-maxdd":                 [RatioMeasure.CALMAR_RATIO, RiskMeasure.MAX_DRAWDOWN],
    "calmar-avgdd":                 [RatioMeasure.CALMAR_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
    "calmar-edar":                  [RatioMeasure.CALMAR_RATIO, RiskMeasure.EDAR],
    "calmar-first-lpm":             [RatioMeasure.CALMAR_RATIO, RiskMeasure.FIRST_LOWER_PARTIAL_MOMENT],
    "calmar-ulcer":                 [RatioMeasure.CALMAR_RATIO, RiskMeasure.ULCER_INDEX],
    "calmar-gini":                  [RatioMeasure.CALMAR_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
}

In [16]:
# Perf + Risk + Ratio
FITNESS_MEASURES = {
    "mean-sortino-ratio-variance":        [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.VARIANCE],
    "mean-sortino-ratio-avgdd":           [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
    "mean-sortino-ratio-cdar":            [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.CDAR],
    "mean-sortino-ratio-cvar":            [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.CVAR],
    "mean-sortino-ratio-edar":            [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.EDAR],
    "mean-sortino-ratio-evar":            [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.EVAR],
    "mean-sortino-ratio-ulcer":           [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.ULCER_INDEX],
    "mean-sortino-ratio-gmd":             [PerfMeasure.MEAN, RatioMeasure.SORTINO_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
    "mean-sharpe-ratio-semivariance":     [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.SEMI_VARIANCE],
    "mean-sharpe-ratio-semivariance":     [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.SEMI_VARIANCE, ExtraRiskMeasure.KURTOSIS],
    "mean-sharpe-ratio-avgdd":            [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN],
    "mean-sharpe-ratio-avgdd-kurt":       [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.AVERAGE_DRAWDOWN, ExtraRiskMeasure.KURTOSIS],
    "mean-sharpe-ratio-cdar":             [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.CDAR],
    "mean-sharpe-ratio-cvar":             [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.CVAR],
    "mean-sharpe-ratio-edar":             [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.EDAR],
    "mean-sharpe-ratio-evar":             [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.EVAR],
    "mean-sharpe-ratio-ulcer":            [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.ULCER_INDEX],
    "mean-sharpe-ratio-gmd":              [PerfMeasure.MEAN, RatioMeasure.SHARPE_RATIO, RiskMeasure.GINI_MEAN_DIFFERENCE],
    "mean-variance-cvar":                 [PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.CVAR],
    "mean-variance-kurt":                 [PerfMeasure.MEAN, RiskMeasure.VARIANCE, ExtraRiskMeasure.KURTOSIS],
    "mean-semivariance-cvar":             [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RiskMeasure.CVAR],
    "mean-variance-avgdd":                [PerfMeasure.MEAN, RiskMeasure.VARIANCE, RiskMeasure.AVERAGE_DRAWDOWN],
    "mean-semivariance-avgdd":            [PerfMeasure.MEAN, RiskMeasure.SEMI_VARIANCE, RiskMeasure.AVERAGE_DRAWDOWN],

}

In [25]:
# ---- 1. 复用你原有的组件 ----
cv = WalkForward(
    test_size=252//2,
    train_size=int(252 * 2),
    purged_size=1,
    reduce_test=True,
    expand_train=False,
)

# ---- 2. 定义 objective 函数 ----
def objective(trial):
    #min_n_assets = trial.suggest_categorical("nondomin__min_n_assets", [5, 10, 15, 20, 25])
    #threshold = trial.suggest_categorical("nondomin__threshold", [-0.25, -0.3, -0.4, -0.5])
    #corr_threshold = trial.suggest_categorical("correlate__threshold", [0.1, 0.2, 0.3, 0.4, 0.5])
    # 有序参数：让 TPE 感知数值距离
    min_n_assets = trial.suggest_int("nondomin__min_n_assets", 5, 25, step=5)
    threshold = trial.suggest_float("nondomin__threshold", -0.5, -0.3, step=0.05)
    corr_threshold = trial.suggest_float("correlate__threshold", 0.1, 0.5, step=0.1)
    # 无序分类参数：fitness_measures 是筛选逻辑，不是目标函数
    fm_name = trial.suggest_categorical("nondomin__fitness_measures", list(FITNESS_MEASURES))
    fitness_measures = FITNESS_MEASURES[fm_name]

    model = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("nondomin", SelectNonDominated(
            min_n_assets=min_n_assets,
            threshold=threshold,
            fitness_measures=fitness_measures,
        )),
        ("correlate", DropCorrelated(threshold=corr_threshold)),
        ("optimization", EqualWeighted()),
    ])

    pred = cross_val_predict(model, X, cv=cv, n_jobs=4)  # 无 y
    return float(pred.annualized_mean)


# ---- 3. 创建 Study 并运行 ----
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(multivariate=True, seed=42),
)

study.optimize(
    objective,
    n_trials=800,       # 等价于你原来的 n_iter=10
    n_jobs=12,          # ⚠️ trial 级并行，每个 trial 内部串行
    show_progress_bar=True,
    callbacks=[StopWhenNoImprovement(patience=100, min_delta=1e-3)],
)

# ---- 4. 查看结果 ----
print("最优参数：", study.best_params)
print("最优得分：", study.best_value)
# 可选：可视化搜索过程
# optuna.visualization.plot_optimization_history(study).show()

g:\Anaconda3\envs\ml4t\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-08-24 18:20:04,132] A new study created in memory with name: no-name-949a3ee2-4fc2-42da-9f9f-17fc3f9369d5


  0%|          | 0/800 [00:00<?, ?it/s]

[I 2026-08-24 18:21:19,948] Trial 1 finished with value: 0.05567924915953604 and parameters: {'nondomin__min_n_assets': 5, 'nondomin__threshold': -0.35, 'correlate__threshold': 0.4, 'nondomin__fitness_measures': 'mean-sharpe-ratio-avgdd-kurt'}. Best is trial 1 with value: 0.05567924915953604.
[I 2026-08-24 18:21:20,476] Trial 2 finished with value: 0.05877246970217201 and parameters: {'nondomin__min_n_assets': 25, 'nondomin__threshold': -0.35, 'correlate__threshold': 0.30000000000000004, 'nondomin__fitness_measures': 'mean-semivariance-cvar'}. Best is trial 2 with value: 0.05877246970217201.
[I 2026-08-24 18:21:22,867] Trial 3 finished with value: 0.04087939496109048 and parameters: {'nondomin__min_n_assets': 20, 'nondomin__threshold': -0.35, 'correlate__threshold': 0.4, 'nondomin__fitness_measures': 'mean-sortino-ratio-cvar'}. Best is trial 2 with value: 0.05877246970217201.
[I 2026-08-24 18:21:23,759] Trial 4 finished with value: 0.04709921315769719 and parameters: {'nondomin__min_n_

#### WalkForward + Best Parameter

In [ ]:
最优参数： {'nondomin__min_n_assets': 5, 'nondomin__threshold': -0.25, 'correlate__threshold': 0.4, 'nondomin__fitness_measures': 'mean-gini-ratio-variance'}
最优得分： 0.0898399265461133
MAX Drawdown                              8.57%
Average Drawdown                          2.34%
Annualized Sharpe Ratio                    1.27
Annualized Sortino Ratio                   1.75

126, 252*2
最优参数： {'nondomin__min_n_assets': 5, 'nondomin__threshold': -0.35, 'correlate__threshold': 0.1, 'nondomin__fitness_measures': 'mean-gini-ratio-semivariance'}
最优得分： 0.10591870482755297
MAX Drawdown                              9.59%
Average Drawdown                          1.83%
Annualized Sharpe Ratio                    1.21
Annualized Sortino Ratio                   1.73

最优参数： {'nondomin__min_n_assets': 15, 'nondomin__threshold': -0.25, 'correlate__threshold': 0.1, 'nondomin__fitness_measures': 'mean-sharpe-ratio-semivariance'}
最优得分： 0.10622994041257404
MAX Drawdown                             17.06%
Average Drawdown                          3.65%
Annualized Sharpe Ratio                    1.19
Annualized Sortino Ratio                   1.69

最优参数： {'nondomin__min_n_assets': 20, 'nondomin__threshold': -0.35, 'correlate__threshold': 0.1, 'nondomin__fitness_measures': 'mean-sharpe-ratio-avgdd'}
最优得分： 0.09875816355755006

#test_size=252//2,train_size=int(252 * 3),
最优参数： {'nondomin__min_n_assets': 5, 'nondomin__threshold': -0.35, 'correlate__threshold': 0.5, 'nondomin__fitness_measures': 'mean-sharpe-ratio-cvar'}
最优得分： 0.09073300958104169
Skew                                    -47.04%
Kurtosis                                798.35%
MAX Drawdown                             11.92%
Average Drawdown                          4.23%
#test_size=252//2,train_size=int(252 * 2),
最优参数： {'nondomin__min_n_assets': 15, 'nondomin__threshold': -0.3, 'correlate__threshold': 0.1, 'nondomin__fitness_measures': 'mean-sortino-ratio-edar'}
最优得分： 0.10564773551492695

In [26]:
selection_pipe = Pipeline([
        ("complete", SelectComplete(drop_assets_with_internal_nan=False)),
        ("variance", DropZeroVariance(threshold=1e-8)),
        ("nondomin", SelectNonDominated(min_n_assets=15, threshold=-0.3,
                        fitness_measures=[PerfMeasure.MEAN,
                        RatioMeasure.SORTINO_RATIO, 
                        RiskMeasure.EDAR])),
        ("correlate", DropCorrelated(threshold=0.1, absolute=False)),
    ])

In [27]:
train_portfolios = []
test_portfolios = []

cv = WalkForward(test_size=252//2, train_size=int(252*2), purged_size=1, reduce_test=True, expand_train=False)
for i, (train_index, test_index) in enumerate(cv.split(X)):
    # 划分训练测试集
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]
    # 完整性筛选
    X_train = selection_pipe.fit_transform(X_train)
    if X_train.empty:
        continue
    # 训练模型
    m = EqualWeighted(portfolio_params=dict(name="Fold %d"%i)).fit(X_train)
    train_portfolios.append(m.predict(X_train))
    test_portfolios.append(m.predict(X_test[X_train.columns]))

population_train = Population(train_portfolios)
population_test = Population(MultiPeriodPortfolio(test_portfolios))

population_train.set_portfolio_params(tag="Train")
population_test.set_portfolio_params(tag="Test")
population = population_train + population_test

In [28]:
population.plot_measures(
    x=RiskMeasure.ANNUALIZED_STANDARD_DEVIATION,
    y=PerfMeasure.ANNUALIZED_MEAN,
    color_scale=RatioMeasure.ANNUALIZED_SHARPE_RATIO,
    hover_measures=[RiskMeasure.MAX_DRAWDOWN, RatioMeasure.ANNUALIZED_SORTINO_RATIO],
)

In [29]:
population_test.plot_cumulative_returns()

In [30]:
MultiPeriodPortfolio(test_portfolios).plot_cumulative_returns()

In [31]:
MultiPeriodPortfolio(test_portfolios).summary()

Mean                                     0.042%
Annualized Mean                          10.56%
Variance                                0.0033%
Annualized Variance                       0.84%
Semi-Variance                           0.0016%
Annualized Semi-Variance                  0.41%
Standard Deviation                        0.58%
Annualized Standard Deviation             9.19%
Semi-Deviation                            0.40%
Annualized Semi-Deviation                 6.43%
Mean Absolute Deviation                   0.35%
CVaR at 95%                               1.33%
EVaR at 95%                               2.95%
Worst Realization                         5.74%
CDaR at 95%                               6.92%
MAX Drawdown                             13.29%
Average Drawdown                          1.97%
EDaR at 95%                               8.68%
First Lower Partial Moment                0.17%
Ulcer Index                               0.026
Gini Mean Difference                    

In [32]:
X_train

,161014.SZ,511030.SH
timestamp,,
2024-04-23 00:00:00+08:00,0.000695,0.000604
2024-04-24 00:00:00+08:00,-0.001389,-0.000431
2024-04-25 00:00:00+08:00,0.000000,0.000000
2024-04-26 00:00:00+08:00,0.000000,-0.000777
2024-04-29 00:00:00+08:00,-0.000696,-0.000950
...,...,...
2026-05-19 00:00:00+08:00,-0.000615,0.000516
2026-05-20 00:00:00+08:00,0.002460,0.000008
2026-05-21 00:00:00+08:00,0.001227,0.000008


In [47]:
X_train

,511220.SH,511360.SH,511810.SH,511880.SH,511900.SH,512770.SH
timestamp,,,,,,
2023-04-07 00:00:00+08:00,0.000226,0.000084,0.00009,0.000101,0.00009,0.008228
2023-04-10 00:00:00+08:00,0.001354,0.000178,0.00006,-0.000054,-0.00009,0.002511
2023-04-11 00:00:00+08:00,-0.000075,0.000084,-0.00015,0.000031,0.00001,-0.006888
2023-04-12 00:00:00+08:00,0.001127,0.000056,0.00014,0.000077,0.00001,0.002522
2023-04-13 00:00:00+08:00,0.000225,0.000094,0.00032,0.000054,0.00001,-0.011321
...,...,...,...,...,...,...
2026-05-19 00:00:00+08:00,0.000275,0.000062,0.00000,0.000067,0.00000,0.006308
2026-05-20 00:00:00+08:00,0.000138,0.000106,0.00001,0.000120,-0.00002,0.016593
2026-05-21 00:00:00+08:00,0.000069,0.000062,0.00006,0.000045,0.00004,-0.020312
